# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIRˆ2) Exploration with `mlcroissant`
This notebook provides a guided workflow for loading and exploring the FAIRˆ2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

The dataset contains detailed clinicopathological and molecular data about cancer survivors who developed second primary colorectal cancer (CRC), supporting investigation of MSI-H status and anatomical distribution among other variables.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant pandas

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}:\n{metadata.description}\n\n")
print(f"Dataset identifier: {metadata.identifier}")
print(f"Authors: {[a for a in metadata.author] if hasattr(metadata, 'author') else 'N/A'}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else 'N/A'}")

## 2. Data Overview
Review available Record Sets, Fields, and their `@id`s as defined in the Croissant schema. Each Record Set and Field is identified by its `@id`, which you can use to reference it in data loading operations.

In [ ]:
# List all available record sets and their fields with @id
record_sets = list(dataset.record_sets)
print(f"Available Record Sets ({len(record_sets)}):\n")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '<no name>')}")
    # List fields for each record set
    fields = rs.get('fields', [])
    if fields:
        print(f"  Fields ({len(fields)}):")
        for field in fields:
            if isinstance(field, dict):
                f_id = field.get('@id', '<no @id>')
                f_name = field.get('name', '<no name>')
            else:
                f_id = field
                f_name = ''
            print(f"    - {f_id} {f'({f_name})' if f_name else ''}")
    else:
        print("  No fields specified.")
    print()
# Save the first record set @id for demo purposes
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]['@id']
else:
    main_record_set_id = None

### View Sample Records
You can iterate through the records in a given record set using the record set's `@id`. Below is a preview of sample records from the primary record set.

In [ ]:
# View a sample of records for the main record set using its @id
if main_record_set_id:
    print(f"Sample records for RecordSet: {main_record_set_id}\n---")
    for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
        pprint.pprint(record)
        if i >= 2:
            break
else:
    print("No record set found.")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Extract data from each record set into DataFrames indexed by RecordSet @id
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    print(f"Loading records for RecordSet @id: {rs_id}")
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df

# Preview columns of the first record set
first_rs_id = main_record_set_id
if first_rs_id and first_rs_id in dataframes:
    print(f"\nFields (columns) in RecordSet {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical analysis steps, such as filtering records based on a numeric field, normalization, and grouping. All references are made by their `@id`s as per Croissant specification.

In [ ]:
# Identify a suitable numeric field @id for analysis.
df = dataframes[first_rs_id]
print(f"Columns in {first_rs_id}:\n", df.columns.tolist())

# Pick a likely numeric field (suppose there is a field like 'age' or 'interval_months')
# We'll attempt to use any suitable column for demonstration
candidate_numeric_field = None
for col in df.columns:
    # Try finding a numeric column by inspecting the first non-null value
    sample = df[col].dropna()
    if not sample.empty and pd.api.types.is_numeric_dtype(sample):
        candidate_numeric_field = col
        break

# Set up for further EDA
if candidate_numeric_field:
    print(f"Using numeric field '@id': {candidate_numeric_field}\n")
    threshold = df[candidate_numeric_field].mean() if df[candidate_numeric_field].mean() > 0 else 10
    filtered_df = df[df[candidate_numeric_field] > threshold]
    print(f"Filtered records with {candidate_numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
    display(filtered_df.head())
    
    # Normalization
    normalized_col = f"{candidate_numeric_field}_normalized"
    filtered_df[normalized_col] = (filtered_df[candidate_numeric_field] - filtered_df[candidate_numeric_field].mean()) / filtered_df[candidate_numeric_field].std()
    print(f"\nNormalized {candidate_numeric_field} for filtered records:")
    display(filtered_df[[candidate_numeric_field, normalized_col]].head())
    
    # Attempt grouping by a non-numeric field
    group_field = None
    for col in df.columns:
        if col != candidate_numeric_field and pd.api.types.is_string_dtype(df[col]):
            group_field = col
            break
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[candidate_numeric_field].mean().to_frame()
        print(f"\nGrouped (by mean) {candidate_numeric_field} by '{group_field}':")
        display(grouped_df.head())
    else:
        print("No suitable categorical field for grouping was found.")
else:
    print("No suitable numeric field found for analysis.")

## 5. Visualization
Visualize data distributions, such as histograms of a selected numeric field or boxplots grouped by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizing the numeric field if available
if candidate_numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[candidate_numeric_field], kde=True, bins=15)
    plt.title(f"Distribution of {candidate_numeric_field}")
    plt.xlabel(candidate_numeric_field)
    plt.ylabel("Count")
    plt.show()

    # Boxplot by group if group_field is found
    if group_field:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field], y=df[candidate_numeric_field])
        plt.title(f"{candidate_numeric_field} by {group_field}")
        plt.xticks(rotation=45, ha='right')
        plt.xlabel(group_field)
        plt.ylabel(candidate_numeric_field)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to load a dataset defined by a Croissant schema, reviewed its record sets and fields using their `@id`s, extracted tabular data for EDA, and visualized key fields. All data elements were referenced by their unique `@id`s, promoting reproducibility and clarity in dataset exploration.

Further analyses can be built by leveraging the semantic structure provided by Croissant schemas for robust data science workflows on FAIR data.